# Open-Domain Claim Verification — API Pipeline

Re-implementation of the full pipeline from [Vladika & Matthes (EACL 2024)](https://arxiv.org/abs/2402.02844), replacing the paper's local knowledge dumps with public APIs.

| Step | Original paper | This notebook |
|---|---|---|
| 1 — Document retrieval | BM25 / BioSimCSE over local dumps | **Wikipedia:** Python `wikipedia` API (keyword search) · **PubMed:** NCBI E-utilities API |
| 2 — Evidence selection | SPICED sentence similarity | Unchanged |
| 3 — Verdict prediction | DeBERTa-v3-large NLI | Unchanged |

Wikipedia API search is keyword-based (comparable to the *Wikipedia BM25* rows in Table 3). NCBI E-utilities queries the same PubMed database as the paper's MEDLINE dump (comparable to *PubMed BM25* rows).

**Estimated runtime on free Google Colab (T4 GPU):**

| Knowledge source | Step 1 — Retrieval | Step 2 — Evidence selection | Step 3 — Verdict prediction |
|---|---|---|---|
| Wikipedia API | ~60–90 min | ~20–30 min | ~15–20 min |
| PubMed API    | ~8 min (with API key) / ~25 min (without) | ~20–30 min | ~15–20 min |

In [3]:
# Installs packages not pre-installed on Colab.
# torch, numpy, pandas, scikit-learn, nltk are already on Colab — excluded to avoid
# overwriting Colab's GPU-enabled PyTorch with a CPU-only version.
# If running locally, install all packages manually:
!pip install wikipedia sentence-transformers transformers torch nltk scikit-learn pandas numpy biopython python-dotenv
!pip install wikipedia sentence-transformers transformers biopython python-dotenv --quiet

## Setup

**Running on Google Colab (VSCode extension or browser):**
1. Run the Drive mount cell below
2. Make sure your Drive has the folder: `My Drive/NLP-Group-Project/`

**Running locally:**
1. Skip the Drive mount cell

In [4]:
# Auto-detects whether running on Google Colab and mounts Drive if so.
# No changes needed — works the same locally and on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running on Colab — Drive mounted.')
except ImportError:
    print('Running locally — skipping Drive mount.')

Mounted at /content/drive
Running on Colab — Drive mounted.


In [8]:
import os
import csv
import time
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch
import wikipedia
import nltk
from Bio import Entrez
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from dotenv import load_dotenv

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    import google.colab
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    BASE_PATH = '/content/drive/MyDrive/NLP-Group-Project'
else:
    BASE_PATH = '../..'

DATASETS_PATH       = os.path.join(BASE_PATH, 'data')
WIKI_RESULTS_PATH   = os.path.join(BASE_PATH, 'reproduction', 'api-pipeline', 'results', 'Wikipedia API')
PUBMED_RESULTS_PATH = os.path.join(BASE_PATH, 'reproduction', 'api-pipeline', 'results', 'PubMed API')
os.makedirs(WIKI_RESULTS_PATH,   exist_ok=True)
os.makedirs(PUBMED_RESULTS_PATH, exist_ok=True)

DATASETS_TO_RUN = ['scifact', 'pubmedqa', 'healthfc', 'covert']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on Colab: {RUNNING_ON_COLAB}')
print(f'Using device:     {device}')
print(f'Datasets path:    {os.path.abspath(DATASETS_PATH)}')
print(f'Wikipedia results:{os.path.abspath(WIKI_RESULTS_PATH)}')
print(f'PubMed results:   {os.path.abspath(PUBMED_RESULTS_PATH)}')

load_dotenv(os.path.join(BASE_PATH, '.env'), override=True)

Running on Colab: True
Using device:     cpu
Datasets path:    /content/drive/MyDrive/NLP-Group-Project/data
Wikipedia results:/content/drive/MyDrive/NLP-Group-Project/reproduction/api-pipeline/results/Wikipedia API
PubMed results:   /content/drive/MyDrive/NLP-Group-Project/reproduction/api-pipeline/results/PubMed API


True

## Step 0 — Load Datasets

Labels are normalised to binary **1 = SUPPORTED**, **0 = REFUTED** for all datasets.
Claims with a NOT ENOUGH INFORMATION label are excluded (consistent with the paper).

| Dataset | Label encoding in CSV | Remapped to |
|---|---|---|
| SciFact | 1 = SUPPORTED, 0 = REFUTED | already 1/0 |
| PubMedQA | 'yes' = SUPPORTED, 'no' = REFUTED, 'maybe' = NEI | 'yes'→1, 'no'→0 |
| HealthFC | 0 = SUPPORTED, 1 = NEI, 2 = REFUTED | 0→1, 2→0 |
| CoVERT | 'SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO' | SUPPORTS→1, REFUTES→0 |

In [6]:
# SciFact — labels already binary: 1=SUPPORTED, 0=REFUTED
scifact_df = pd.read_csv(os.path.join(DATASETS_PATH, 'scifact_no-nei_dataset.csv'), index_col=[0])
scifact_claims = scifact_df.claim.tolist()
scifact_labels = scifact_df.label.tolist()

# PubMedQA — exclude 'maybe', map yes→1, no→0
pubmedqa_df = pd.read_json(os.path.join(DATASETS_PATH, 'pubmedqa.json')).transpose()
pubmedqa_df = pubmedqa_df[pubmedqa_df.final_decision.isin(['yes', 'no'])]
pubmedqa_claims = pubmedqa_df.QUESTION.tolist()
pubmedqa_labels = [1 if l == 'yes' else 0 for l in pubmedqa_df.final_decision.tolist()]

# HealthFC — exclude NEI (label=1), remap 0→1 (SUPPORTED), 2→0 (REFUTED)
healthfc_df = pd.read_csv(os.path.join(DATASETS_PATH, 'healthFC_annotated.csv'))
healthfc_df = healthfc_df[healthfc_df.label != 1].copy()
healthfc_claims = healthfc_df.en_claim.tolist()
healthfc_labels = [1 if l == 0 else 0 for l in healthfc_df.label.tolist()]

# CoVERT — exclude 'NOT ENOUGH INFO', map SUPPORTS→1, REFUTES→0
covert_df = pd.read_json(os.path.join(DATASETS_PATH, 'CoVERT_FC_annotations.jsonl'), lines=True)
covert_df = covert_df[covert_df.label != 'NOT ENOUGH INFO'].copy()
covert_df['claim'] = covert_df.claim.str.replace('@username', '', regex=False).str.replace('\n', ' ', regex=False)
covert_claims = covert_df.claim.tolist()
covert_labels = [1 if l == 'SUPPORTS' else 0 for l in covert_df.label.tolist()]

all_datasets = {
    'scifact':  (scifact_claims,  scifact_labels),
    'pubmedqa': (pubmedqa_claims, pubmedqa_labels),
    'healthfc': (healthfc_claims, healthfc_labels),
    'covert':   (covert_claims,   covert_labels),
}

print('Dataset sizes (after excluding NEI):')
for name, (claims, labels) in all_datasets.items():
    n_pos = sum(labels)
    print(f'  {name:<12}: {len(claims):>4} claims | {n_pos} supported, {len(labels)-n_pos} refuted')

Dataset sizes (after excluding NEI):
  scifact     :  693 claims | 456 supported, 237 refuted
  pubmedqa    :  890 claims | 552 supported, 338 refuted
  healthfc    :  327 claims | 202 supported, 125 refuted
  covert      :  264 claims | 198 supported, 66 refuted


## Step 1 — Document Retrieval (Wikipedia API)

For each claim, `wikipedia.search()` returns the titles of the most relevant Wikipedia articles (keyword-based, similar to BM25). We then fetch the full text of each article and collect it as our evidence pool.

A 0.3-second sleep between page fetches keeps us within Wikipedia's rate limits.

In [5]:
def retrieve_wikipedia_articles(claim, n_results=10):
    """
    Search Wikipedia for a claim and return:
      - fetched_titles: list of article titles (Wikipedia equivalent of PMIDs)
      - articles:       list of article text content
    Silently skips disambiguation pages and pages that cannot be fetched.
    """
    try:
        titles = wikipedia.search(claim, results=n_results)
    except Exception:
        return [], []

    fetched_titles = []
    articles = []
    for title in titles:
        try:
            page = wikipedia.page(title, auto_suggest=False)
            fetched_titles.append(page.title)
            articles.append(page.content)
        except (wikipedia.exceptions.DisambiguationError,
                wikipedia.exceptions.PageError,
                Exception):
            continue
        time.sleep(0.3)
    return fetched_titles, articles

In [6]:
retrieved_titles   = {}
retrieved_articles = {}

for dataset_name in DATASETS_TO_RUN:
    claims, _ = all_datasets[dataset_name]
    print(f'\nRetrieving Wikipedia articles for {dataset_name} ({len(claims)} claims)...')
    titles_per_claim   = []
    articles_per_claim = []

    for i, claim in enumerate(claims):
        titles, articles = retrieve_wikipedia_articles(claim)
        titles_per_claim.append(titles)
        articles_per_claim.append(articles)
        if (i + 1) % 100 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    retrieved_titles[dataset_name]   = titles_per_claim
    retrieved_articles[dataset_name] = articles_per_claim
    avg = np.mean([len(a) for a in articles_per_claim])
    print(f'  Done. Average articles retrieved per claim: {avg:.1f}')


Retrieving Wikipedia articles for scifact (693 claims)...
  100/693 claims processed
  200/693 claims processed
  300/693 claims processed
  400/693 claims processed
  500/693 claims processed
  600/693 claims processed
  693/693 claims processed
  Done. Average articles retrieved per claim: 0.1

Retrieving Wikipedia articles for pubmedqa (890 claims)...
  100/890 claims processed
  200/890 claims processed
  300/890 claims processed
  400/890 claims processed
  500/890 claims processed
  600/890 claims processed
  700/890 claims processed
  800/890 claims processed
  890/890 claims processed
  Done. Average articles retrieved per claim: 0.1

Retrieving Wikipedia articles for healthfc (327 claims)...
  100/327 claims processed
  200/327 claims processed
  300/327 claims processed
  327/327 claims processed
  Done. Average articles retrieved per claim: 0.1

Retrieving Wikipedia articles for covert (264 claims)...
  100/264 claims processed
  200/264 claims processed
  264/264 claims pr

In [ ]:
for dataset_name in DATASETS_TO_RUN:
    claims, _ = all_datasets[dataset_name]
    filepath = os.path.join(WIKI_RESULTS_PATH, f'{dataset_name}_wiki_api_titles.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for claim, titles in zip(claims, retrieved_titles[dataset_name]):
            f.write(claim + '\n')
            f.write(str(titles) + '\n')
            f.write('\n')
    print(f'Saved: {filepath}')

## Step 2 — Evidence Selection (SPICED)

All sentences from the retrieved articles are scored against the claim using the **SPICED** sentence similarity model ([Wright et al., 2022](https://aclanthology.org/2022.emnlp-main.117.pdf)), which was shown to work well for paraphrases of scientific claims.

The top 10 sentences (by cosine similarity) are concatenated with the claim to form the input for the NLI step:

```
claim [SEP] evidence_sentence_1 evidence_sentence_2 ... evidence_sentence_10
```

In [8]:
EVIDENCE_MODEL_NAME = 'copenlu/spiced'
print(f'Loading evidence selection model: {EVIDENCE_MODEL_NAME}')
evidence_model = SentenceTransformer(EVIDENCE_MODEL_NAME)


def select_top_sentences(claim, articles, model, top_k=10):
    """
    Extract all sentences from retrieved articles, then select the top-k
    most similar to the claim using cosine similarity.
    Returns a single string of the selected sentences.
    """
    all_sentences = []
    for article in articles:
        sentences = sent_tokenize(article)
        all_sentences.extend([s.lower() for s in sentences])

    if not all_sentences:
        return ''

    top_k = min(top_k, len(all_sentences))
    sents_embeddings = model.encode(all_sentences, convert_to_tensor=True)
    claim_embedding  = model.encode(claim, convert_to_tensor=True)
    cos_scores = util.cos_sim(claim_embedding, sents_embeddings)[0]

    top_indices = torch.topk(cos_scores, k=top_k).indices.cpu().numpy()
    top_indices = np.sort(top_indices)
    selected = np.array(all_sentences)[top_indices]
    return ' '.join(selected)

Loading evidence selection model: copenlu/spiced


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
joint_lists = {}

for dataset_name in DATASETS_TO_RUN:
    claims, _ = all_datasets[dataset_name]
    articles_list = retrieved_articles[dataset_name]
    print(f'\nSelecting evidence sentences for {dataset_name}...')

    joint = []
    for i, (claim, articles) in enumerate(zip(claims, articles_list)):
        evidence_str = select_top_sentences(claim, articles, evidence_model)
        joint.append(f'{claim} [SEP] {evidence_str}')
        if (i + 1) % 100 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    joint_lists[dataset_name] = joint
    print('  Done.')


Selecting evidence sentences for scifact...
  100/693 claims processed
  200/693 claims processed
  300/693 claims processed
  400/693 claims processed
  500/693 claims processed
  600/693 claims processed
  693/693 claims processed
  Done.

Selecting evidence sentences for pubmedqa...
  100/890 claims processed
  200/890 claims processed
  300/890 claims processed
  400/890 claims processed
  500/890 claims processed
  600/890 claims processed
  700/890 claims processed
  800/890 claims processed
  890/890 claims processed
  Done.

Selecting evidence sentences for healthfc...
  100/327 claims processed
  200/327 claims processed
  300/327 claims processed
  327/327 claims processed
  Done.

Selecting evidence sentences for covert...
  100/264 claims processed
  200/264 claims processed
  264/264 claims processed
  Done.


In [ ]:
for dataset_name in DATASETS_TO_RUN:
    filepath = os.path.join(WIKI_RESULTS_PATH, f'{dataset_name}_wiki_api_lines.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for line in joint_lists[dataset_name]:
            f.write(line + '\n')
    print(f'Saved: {filepath}')

In [ ]:
# ─── RESUME POINT (Wikipedia) ────────────────────────────────────────────────
# After a session restart, run this cell (plus Drive mount, imports, load
# datasets) to reload saved Wikipedia joint lines and skip retrieval + SPICED.
# ─────────────────────────────────────────────────────────────────────────────
if 'joint_lists' not in vars() or not joint_lists:
    joint_lists = {}
    all_loaded = True
    for dataset_name in DATASETS_TO_RUN:
        filepath = os.path.join(WIKI_RESULTS_PATH, f'{dataset_name}_wiki_api_lines.txt')
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                joint_lists[dataset_name] = [line.rstrip('\n') for line in f]
            print(f'Loaded {dataset_name}: {len(joint_lists[dataset_name])} lines')
        else:
            print(f'No saved lines for {dataset_name} — run Wikipedia retrieval + SPICED first.')
            all_loaded = False
    if all_loaded:
        print('All Wikipedia joint lines loaded. Ready for verdict prediction.')
else:
    print('Wikipedia joint lines already in memory.')

In [ ]:
NLI_MODEL_NAME = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'
print(f'Loading NLI model: {NLI_MODEL_NAME}')
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME, model_max_length=1024)
nli_model     = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
print('NLI model loaded.')

---
## Step 3 — Verdict Prediction (DeBERTa-v3-large NLI)

The **DeBERTa-v3-large** model fine-tuned on MNLI/FEVER/ANLI predicts whether the evidence *entails* (SUPPORTED) or *contradicts* (REFUTED) the claim.

- P(entailment) > P(contradiction) → **SUPPORTED** (1)
- Otherwise → **REFUTED** (0)

In [ ]:
class ClaimDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: v[idx].clone().detach() for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


def predict_verdicts(joint_list, model, tokenizer, batch_size=16):
    encodings = tokenizer(
        joint_list, return_tensors='pt', truncation='only_first',
        padding=True, max_length=1024
    )
    dataset = ClaimDataset(encodings, [0] * len(joint_list))
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()
    model.to(device)
    predictions = []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )[0]
            probs = logits.softmax(dim=1)
            predictions.extend((probs[:, 0] > probs[:, 2]).cpu().numpy().astype(int).tolist())
    return np.array(predictions)


def save_metrics(results, save_path):
    with open(save_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['dataset', 'precision', 'recall', 'f1'])
        writer.writeheader()
        for ds, s in results.items():
            writer.writerow({
                'dataset':   ds,
                'precision': round(s['precision'] * 100, 1),
                'recall':    round(s['recall']    * 100, 1),
                'f1':        round(s['f1']        * 100, 1),
            })
    print(f'Saved: {save_path}')


def load_metrics(path):
    results = {}
    with open(path, 'r', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            results[row['dataset']] = {
                'precision': float(row['precision']) / 100,
                'recall':    float(row['recall'])    / 100,
                'f1':        float(row['f1'])        / 100,
            }
    return results


print('Helper functions defined.')

In [ ]:
wiki_results = {}

print(f'{"Dataset":<12} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 45)

for dataset_name in DATASETS_TO_RUN:
    _, gold_labels = all_datasets[dataset_name]
    predictions = predict_verdicts(joint_lists[dataset_name], nli_model, nli_tokenizer)
    p = precision_score(gold_labels, predictions, average='binary', zero_division=0)
    r = recall_score(   gold_labels, predictions, average='binary', zero_division=0)
    f = f1_score(       gold_labels, predictions, average='binary', zero_division=0)
    wiki_results[dataset_name] = {'precision': p, 'recall': r, 'f1': f}
    print(f'{dataset_name:<12} {p*100:>10.1f} {r*100:>10.1f} {f*100:>10.1f}')

print('-' * 45)
print('(compare against Table 3 Wikipedia BM25 rows in the paper)')

In [ ]:
save_metrics(wiki_results, os.path.join(WIKI_RESULTS_PATH, 'metrics.csv'))

In [ ]:
# ─── RESUME POINT (metrics) ──────────────────────────────────────────────────
# Run this cell after Drive mount + imports + load datasets to reload saved
# metrics without re-running retrieval or verdict prediction.
# ─────────────────────────────────────────────────────────────────────────────

if 'wiki_results' not in vars() or not wiki_results:
    _path = os.path.join(WIKI_RESULTS_PATH, 'metrics.csv')
    if os.path.exists(_path):
        wiki_results = load_metrics(_path)
        print('Loaded Wikipedia API metrics.')
        for ds, s in wiki_results.items():
            print(f'  {ds:<12}: F1={s["f1"]*100:.1f}')
    else:
        wiki_results = {}
        print('No saved Wikipedia API metrics — run the Wikipedia pipeline first.')
else:
    print('Wikipedia API results already in memory.')

if 'pubmed_results' not in vars() or not pubmed_results:
    _path = os.path.join(PUBMED_RESULTS_PATH, 'metrics.csv')
    if os.path.exists(_path):
        pubmed_results = load_metrics(_path)
        print('Loaded PubMed API metrics.')
        for ds, s in pubmed_results.items():
            print(f'  {ds:<12}: F1={s["f1"]*100:.1f}')
    else:
        pubmed_results = {}
        print('No saved PubMed API metrics — run the PubMed pipeline first.')
else:
    print('PubMed API results already in memory.')

---

## PubMed API (Table 3 equivalent)

Replicates the PubMed knowledge source from the paper using the **NCBI E-utilities API** (via Biopython), replacing the original's local 20.6M-abstract MEDLINE dump.

NCBI E-utilities searches the same PubMed database using a keyword index — directly comparable to the **PubMed BM25** rows in Table 3.

Steps 2 (SPICED evidence selection) and 3 (DeBERTa NLI) are identical to the Wikipedia API section above.

> **Rate limit:** Free tier allows 3 requests/second. With a free NCBI API key the limit rises to 10 req/second. Running all 4 datasets (~2,174 claims × 2 requests each) takes ~25 minutes at 3 req/sec, ~8 minutes with an API key.
>
> **Setup:** Provide any valid email address in `Entrez.email` (required by NCBI). Optionally set `Entrez.api_key` for higher throughput.

In [9]:
Entrez.email = os.getenv('NCBI_EMAIL', 'your@email.com')
ncbi_api_key = os.getenv('NCBI_API_KEY', '')
if ncbi_api_key:
    Entrez.api_key = ncbi_api_key

print(f'NCBI email: {Entrez.email}')
print(f'NCBI API key set: {bool(ncbi_api_key)}')
print(f'PubMed results:   {os.path.abspath(PUBMED_RESULTS_PATH)}')

NCBI email: lea.sarouphim@gmail.com
NCBI API key set: True
PubMed results:   /content/drive/MyDrive/NLP-Group-Project/reproduction/api-pipeline/results/PubMed API


In [10]:
def retrieve_pubmed_articles(claim, n_results=10):
    """
    Search PubMed for a claim via NCBI E-utilities and return:
      - fetched_pmids: list of PubMed IDs (equivalent to Wikipedia titles)
      - articles:      list of 'title + abstract' strings
    Mirrors retrieve_wikipedia_articles() in structure.
    """
    try:
        search_handle = Entrez.esearch(db='pubmed', term=claim, retmax=n_results)
        search_record = Entrez.read(search_handle)
        search_handle.close()
        pmids = search_record['IdList']

        if not pmids:
            return [], []

        time.sleep(0.1 if ncbi_api_key else 0.34)  # 10 req/sec with key, 3 req/sec without

        fetch_handle = Entrez.efetch(db='pubmed', id=pmids, rettype='xml', retmode='xml')
        fetch_data   = fetch_handle.read()
        fetch_handle.close()

        root = ET.fromstring(fetch_data)
        fetched_pmids = []
        articles      = []

        for article in root.findall('.//PubmedArticle'):
            pmid_el  = article.find('.//PMID')
            pmid     = pmid_el.text if pmid_el is not None else ''

            title_el = article.find('.//ArticleTitle')
            title    = (title_el.text or '') if title_el is not None else ''

            abstract_els = article.findall('.//AbstractText')
            abstract = ' '.join(el.text or '' for el in abstract_els if el.text)

            if abstract:
                fetched_pmids.append(pmid)
                articles.append(f'{title} {abstract}')

        return fetched_pmids, articles

    except Exception as e:
        print(f'Error: {e}')
        return [], []

In [ ]:
retrieved_pmids           = {}
retrieved_pubmed_articles = {}

for dataset_name in DATASETS_TO_RUN:
    claims, _ = all_datasets[dataset_name]
    print(f'\nRetrieving PubMed abstracts for {dataset_name} ({len(claims)} claims)...')
    pmids_per_claim    = []
    articles_per_claim = []

    for i, claim in enumerate(claims):
        pmids, articles = retrieve_pubmed_articles(claim)
        pmids_per_claim.append(pmids)
        articles_per_claim.append(articles)
        if (i + 1) % 100 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    retrieved_pmids[dataset_name]           = pmids_per_claim
    retrieved_pubmed_articles[dataset_name] = articles_per_claim
    avg = np.mean([len(a) for a in articles_per_claim])
    print(f'  Done. Average abstracts retrieved per claim: {avg:.1f}')

# Save retrieved PMIDs — same format as *_wiki_api_titles.txt
for dataset_name in DATASETS_TO_RUN:
    claims, _ = all_datasets[dataset_name]
    filepath = os.path.join(PUBMED_RESULTS_PATH, f'{dataset_name}_pubmed_api_pmids.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for claim, pmids in zip(claims, retrieved_pmids[dataset_name]):
            f.write(claim + '\n')
            f.write(str(pmids) + '\n')
            f.write('\n')
    print(f'Saved: {filepath}')


Retrieving PubMed abstracts for scifact (693 claims)...
  100/693 claims processed
  200/693 claims processed
  300/693 claims processed
  400/693 claims processed
  600/693 claims processed
  693/693 claims processed
  Done. Average abstracts retrieved per claim: 2.6

Retrieving PubMed abstracts for pubmedqa (890 claims)...
  100/890 claims processed
  200/890 claims processed
  300/890 claims processed


In [ ]:
# Step 2 — SPICED evidence selection (reuses evidence_model already loaded above)
pubmed_joint_lists = {}

for dataset_name in DATASETS_TO_RUN:
    claims, _ = all_datasets[dataset_name]
    articles_list = retrieved_pubmed_articles[dataset_name]
    print(f'\nSelecting evidence sentences for {dataset_name}...')

    joint = []
    for i, (claim, articles) in enumerate(zip(claims, articles_list)):
        evidence_str = select_top_sentences(claim, articles, evidence_model)
        joint.append(f'{claim} [SEP] {evidence_str}')
        if (i + 1) % 100 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    pubmed_joint_lists[dataset_name] = joint
    print('  Done.')

# Save joint lines — same format as *_wiki_api_lines.txt
for dataset_name in DATASETS_TO_RUN:
    filepath = os.path.join(PUBMED_RESULTS_PATH, f'{dataset_name}_pubmed_api_lines.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for line in pubmed_joint_lists[dataset_name]:
            f.write(line + '\n')
    print(f'Saved: {filepath}')

In [ ]:
# ─── RESUME POINT (PubMed) ───────────────────────────────────────────────────
# After a session restart, run this cell (plus Drive mount, imports, load
# datasets, and the PubMed setup cell above) to reload saved joint lines from
# Drive and skip PubMed retrieval + SPICED evidence selection entirely.
# ─────────────────────────────────────────────────────────────────────────────
if 'pubmed_joint_lists' not in vars() or not pubmed_joint_lists:
    pubmed_joint_lists = {}
    all_loaded = True
    for dataset_name in DATASETS_TO_RUN:
        filepath = os.path.join(PUBMED_RESULTS_PATH, f'{dataset_name}_pubmed_api_lines.txt')
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                pubmed_joint_lists[dataset_name] = [line.rstrip('\n') for line in f]
            print(f'Loaded {dataset_name}: {len(pubmed_joint_lists[dataset_name])} lines')
        else:
            print(f'No saved lines for {dataset_name} — run PubMed retrieval + SPICED first.')
            all_loaded = False
    if all_loaded:
        print('All PubMed joint lines loaded. Ready for verdict prediction.')
else:
    print('PubMed joint lines already in memory.')

In [ ]:
pubmed_results = {}

print(f'{"Dataset":<12} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 45)

for dataset_name in DATASETS_TO_RUN:
    _, gold_labels = all_datasets[dataset_name]
    predictions = predict_verdicts(pubmed_joint_lists[dataset_name], nli_model, nli_tokenizer)
    p = precision_score(gold_labels, predictions, average='binary', zero_division=0)
    r = recall_score(   gold_labels, predictions, average='binary', zero_division=0)
    f = f1_score(       gold_labels, predictions, average='binary', zero_division=0)
    pubmed_results[dataset_name] = {'precision': p, 'recall': r, 'f1': f}
    print(f'{dataset_name:<12} {p*100:>10.1f} {r*100:>10.1f} {f*100:>10.1f}')

print('-' * 45)
print('(compare against Table 3 PubMed BM25 rows in the paper)')

save_metrics(pubmed_results, os.path.join(PUBMED_RESULTS_PATH, 'metrics.csv'))

In [ ]:
# F1 comparison: paper's BM25 baselines vs. our API-based results
# Paper values from Table 3 of Vladika & Matthes (EACL 2024)
paper_wiki_bm25   = {'scifact': 74.8, 'pubmedqa': 73.1, 'healthfc': 73.1, 'covert': 75.2}
paper_pubmed_bm25 = {'scifact': 76.1, 'pubmedqa': 70.3, 'healthfc': 69.7, 'covert': 79.5}

header = f'{"Dataset":<12} {"Paper Wiki BM25":>16} {"Our Wiki API":>14} {"Paper PubMed BM25":>18} {"Our PubMed API":>16}'
print(header)
print('-' * len(header))

for ds in DATASETS_TO_RUN:
    pw = paper_wiki_bm25.get(ds,   float('nan'))
    ow = wiki_results[ds]['f1']   * 100
    pp = paper_pubmed_bm25.get(ds, float('nan'))
    op = pubmed_results[ds]['f1'] * 100
    print(f'{ds:<12} {pw:>16.1f} {ow:>14.1f} {pp:>18.1f} {op:>16.1f}')

print('-' * len(header))
print('All scores are F1 (%).')